# 1 Install Required Libraries

In [100]:
%%capture
!apt-get install -y poppler-utils tesseract-ocr
!pip install langchain langchain-community sentence-transformers faiss-cpu transformers accelerate pypdf pdf2image pytesseract langchain-text-splitters

# 2 Import Libraries

In [101]:
# General utility and OS interaction
import os
import shutil # Added for path diagnostics

# Document loading
from langchain_community.document_loaders import PyPDFLoader

# Text splitting
# Updated import for RecursiveCharacterTextSplitter as it's now in langchain_text_splitters
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Embeddings
from langchain_community.embeddings import HuggingFaceEmbeddings

# Vector store
from langchain_community.vectorstores import FAISS

# LLM
from transformers import pipeline, AutoTokenizer, AutoModelForSeq2SeqLM

# For OCR fallback
from pdf2image import convert_from_path
import pytesseract
from PIL import Image

# For LangChain Document object
from langchain_core.documents import Document # Added for consistent Document class

# For displaying outputs
from IPython.display import display, Markdown

# Suppress warnings
import warnings
warnings.filterwarnings('ignore')

# --- Global Configuration for OCR Tools ---
# Set pytesseract command path for robustness
pytesseract.pytesseract.tesseract_cmd = '/usr/bin/tesseract'

# Ensure /usr/bin is in PATH for poppler utilities (like pdfinfo) and tesseract
# This is crucial for pdf2image and pytesseract to find their executables.
# Unconditionally prepend '/usr/bin' to ensure it's at the beginning.
os.environ['PATH'] = '/usr/bin' + os.pathsep + os.environ['PATH']

# 3 Load PDF

In [115]:
# Define the path to the uploaded PDF
pdf_path = '/content/Int-M.Sc-Information-Brochure-2026.pdf'

# Define the path to poppler-utils binaries (common for Colab after apt-get install)
poppler_path = '/usr/bin'

def load_pdf_with_ocr_fallback(pdf_path):
    # Dictionary to store the best extracted Document for each page index
    extracted_content_by_page = {}
    total_pages_identified = 0

    # 1. Attempt PyPDFLoader first
    print("Attempting to load PDF with PyPDFLoader...")
    try:
        temp_py_pdf_docs = PyPDFLoader(pdf_path).load()
        for doc in temp_py_pdf_docs:
            # PyPDFLoader often uses 0-indexed page numbers directly from metadata
            page_idx = doc.metadata.get('page')
            if page_idx is not None and doc.page_content.strip():
                # Store PyPDFLoader's result, will be overwritten by OCR if OCR is better for this page
                extracted_content_by_page[page_idx] = Document(page_content=doc.page_content, metadata={'source': pdf_path, 'page': page_idx})
        print(f"PyPDFLoader extracted content for {len(extracted_content_by_page)} unique pages.")
    except Exception as e:
        print(f"Error with PyPDFLoader: {e}")

    # 2. Attempt OCR for all pages to ensure comprehensive coverage, possibly overriding PyPDFLoader results
    print("\n--- Attempting OCR for all pages ---")
    images = []
    try:
        print("Converting PDF pages to images for OCR...")
        images = convert_from_path(pdf_path, poppler_path=poppler_path)
        total_pages_identified = len(images)
        print(f"Successfully converted {total_pages_identified} pages to images for OCR.")
    except Exception as e:
        print(f"Error converting PDF to images for OCR: {e}. OCR cannot proceed.")
        images = [] # Ensure images is empty if conversion failed

    if images:
        ocr_pages_with_content = 0
        for i, image in enumerate(images):
            try:
                text = pytesseract.image_to_string(image)
                if text.strip():
                    # OCR results are generally preferred as they capture text even from image-based PDFs.
                    # This explicitly overwrites PyPDFLoader content for the same page if OCR finds something.
                    extracted_content_by_page[i] = Document(page_content=text, metadata={'source': pdf_path, 'page': i})
                    print(f"  OCR: Page {i+1} extracted {len(text.strip())} characters.")
                    ocr_pages_with_content += 1
                else:
                    print(f"  OCR: Page {i+1} extracted no meaningful text (length 0 or whitespace).")
            except Exception as ocr_e:
                print(f"OCR failed for page {i+1}: {ocr_e}")
        print(f"OCR processed {total_pages_identified} image pages, found content in {ocr_pages_with_content} of them.")
    else:
        print("No images were processed by OCR.")

    # Consolidate and sort documents by page number
    # If total_pages_identified is available from images, iterate up to that to ensure all pages are considered,
    # even if no content was found by either method.
    final_docs = []
    for i in range(total_pages_identified):
        if i in extracted_content_by_page:
            final_docs.append(extracted_content_by_page[i])
        # Optional: Add placeholder if a page was entirely empty, for consistent page count
        # else:
        #    final_docs.append(Document(page_content="", metadata={'source': pdf_path, 'page': i, 'empty': True}))

    # Fallback if no images were processed at all but PyPDFLoader found something
    if not final_docs and extracted_content_by_page:
        print("No images processed, but PyPDFLoader found content. Using PyPDFLoader results.")
        final_docs = [extracted_content_by_page[page_idx] for page_idx in sorted(extracted_content_by_page.keys())]

    # Diagnostic prints at the end
    if not final_docs:
        print("No documents could be loaded from the PDF.")
        return []
    else:
        print(f"\nFinal count of extracted documents: {len(final_docs)}")
        print(f"--- Diagnostic: Content of first document (page_content length {len(final_docs[0].page_content)}): ---")
        print(final_docs[0].page_content[:1000]) # Print more to see if it's empty
        print("-----------------------------------------------------------------------------------------------------")
        display(Markdown(f"### Sample of First Document (Page 0):\nSource: {final_docs[0].metadata.get('source')}, Page: {final_docs[0].metadata.get('page')}\n\n```text\n{final_docs[0].page_content[:500]}...\n```"))
        return final_docs

# Load the PDF using the function
all_documents = load_pdf_with_ocr_fallback(pdf_path)


Attempting to load PDF with PyPDFLoader...
PyPDFLoader extracted content for 1 unique pages.

--- Attempting OCR for all pages ---
Converting PDF pages to images for OCR...
Error converting PDF to images for OCR: Unable to get page count. Is poppler installed and in PATH?. OCR cannot proceed.
No images were processed by OCR.
No images processed, but PyPDFLoader found content. Using PyPDFLoader results.

Final count of extracted documents: 1
--- Diagnostic: Content of first document (page_content length 1126): ---
VIT
® 
Vellore Institute of Technology lTntegrated M.Sc. Programmes 2026-27
(Deemed to be Universi1y under section 3 of UGC Ace, 1956) 
14. IMPORTANT DATES
Issue of Online Application 
Last date to apply 
28th January, 2026 (Wednesday) 
27th May, 2026 (Wednesday)
Result & Programme Choice preference 9th to 11th June, 2026 (Tuesday to Thursday)
Seat Allotments 17th June, 2026 (Wednesday) 
Last date of payment of Tuition fees 25th June, 2026 (Thursday) 
Orientation Program July,

### Sample of First Document (Page 0):
Source: /content/Int-M.Sc-Information-Brochure-2026.pdf, Page: 14

```text
VIT
® 
Vellore Institute of Technology lTntegrated M.Sc. Programmes 2026-27
(Deemed to be Universi1y under section 3 of UGC Ace, 1956) 
14. IMPORTANT DATES
Issue of Online Application 
Last date to apply 
28th January, 2026 (Wednesday) 
27th May, 2026 (Wednesday)
Result & Programme Choice preference 9th to 11th June, 2026 (Tuesday to Thursday)
Seat Allotments 17th June, 2026 (Wednesday) 
Last date of payment of Tuition fees 25th June, 2026 (Thursday) 
Orientation Program July, 2026 
Commencement...
```

# 4 Text Chunking

In [116]:
# Initialize the RecursiveCharacterTextSplitter
# This splitter tries to split along paragraphs, then sentences, then words.
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,       # The maximum size of each chunk
    chunk_overlap=150,    # The amount of overlap between adjacent chunks
    length_function=len,  # Function to calculate chunk length (using character count)
    add_start_index=True  # Adds metadata about the start index of the chunk
)

# print(f"Text splitter initialized with chunk_size={text_splitter.chunk_size}, chunk_overlap={text_splitter.chunk_overlap}") # Removed due to AttributeError

# Diagnostic: Check all_documents before splitting
print(f"Number of documents in all_documents: {len(all_documents)}")
if all_documents:
    print(f"First document page_content length: {len(all_documents[0].page_content)}")
    # print(f"First document page_content (first 500 chars):\n{all_documents[0].page_content[:500]}...")
else:
    print("all_documents is empty, cannot create chunks.")

# Diagnostic: Test splitter with a simple long string
test_long_string = "This is a very very long string that should be split into multiple chunks by the text splitter. " * 20
print(f"Test string length: {len(test_long_string)}")
test_chunks_result = text_splitter.split_text(test_long_string)
print(f"Number of chunks from simple test string: {len(test_chunks_result)}")
if test_chunks_result:
    print(f"Sample test chunk length: {len(test_chunks_result[0])}")


# Split the loaded documents into chunks
chunks = text_splitter.split_documents(all_documents)

# Print the number of chunks created
print(f"\nNumber of chunks created from all_documents: {len(chunks)}")

# Display a sample chunk to verify
if chunks:
    display(Markdown(f"### Sample of a Text Chunk:\nSource: {chunks[0].metadata.get('source')}, Page: {chunks[0].metadata.get('page')}, Start Index: {chunks[0].metadata.get('start_index')}\n\n```text\n{chunks[0].page_content[:500]}...\n```"))
else:
    print("No chunks were created from all_documents.")


Number of documents in all_documents: 1
First document page_content length: 1126
Test string length: 1920
Number of chunks from simple test string: 3
Sample test chunk length: 799

Number of chunks created from all_documents: 2


### Sample of a Text Chunk:
Source: /content/Int-M.Sc-Information-Brochure-2026.pdf, Page: 14, Start Index: 0

```text
VIT
® 
Vellore Institute of Technology lTntegrated M.Sc. Programmes 2026-27
(Deemed to be Universi1y under section 3 of UGC Ace, 1956) 
14. IMPORTANT DATES
Issue of Online Application 
Last date to apply 
28th January, 2026 (Wednesday) 
27th May, 2026 (Wednesday)
Result & Programme Choice preference 9th to 11th June, 2026 (Tuesday to Thursday)
Seat Allotments 17th June, 2026 (Wednesday) 
Last date of payment of Tuition fees 25th June, 2026 (Thursday) 
Orientation Program July, 2026 
Commencement...
```

# 5 Generate Embeddings

In [104]:
# Initialize HuggingFaceEmbeddings with the specified model
# 'sentence-transformers/all-MiniLM-L6-v2' is a good general-purpose model for sentence embeddings
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

print("HuggingFaceEmbeddings model 'all-MiniLM-L6-v2' initialized.")

# To verify, let's generate an embedding for a small piece of text
if chunks:
    sample_text = chunks[0].page_content[:100]
    sample_embedding = embeddings.embed_query(sample_text)
    print(f"\nSample text for embedding: '{sample_text}'")
    print(f"Generated embedding dimension: {len(sample_embedding)}")
    # print(f"First 5 values of sample embedding: {sample_embedding[:5]}...")
else:
    print("No chunks available to generate sample embedding.")


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

HuggingFaceEmbeddings model 'all-MiniLM-L6-v2' initialized.

Sample text for embedding: 'VIT
® 
Vellore Institute of Technology lTntegrated M.Sc. Programmes 2026-27
(Deemed to be Universi1y'
Generated embedding dimension: 384


# 6 Create FAISS Vector Database

In [117]:
# Create a FAISS vector database from the document chunks and their embeddings
# This step will take the list of 'chunks' and use the 'embeddings' model
# to convert each chunk into a vector and store it in FAISS for efficient similarity search.
print("Creating FAISS vector database... This may take a moment.")
db = FAISS.from_documents(chunks, embeddings)

print("FAISS vector database created successfully!")

# You can optionally perform a simple similarity search to confirm it works
if db:
    test_query = "What are the admission requirements?"
    docs = db.similarity_search(test_query, k=1)
    print(f"\nSuccessfully performed a test similarity search for: '{test_query}'")
    if docs:
        display(Markdown(f"### Top result from FAISS:\nSource: {docs[0].metadata.get('source')}, Page: {docs[0].metadata.get('page')}\n\n```text\n{docs[0].page_content[:300]}...\n```"))
    else:
        print("No results found for test query.")


Creating FAISS vector database... This may take a moment.
FAISS vector database created successfully!

Successfully performed a test similarity search for: 'What are the admission requirements?'


### Top result from FAISS:
Source: /content/Int-M.Sc-Information-Brochure-2026.pdf, Page: 14

```text
VIT
® 
Vellore Institute of Technology lTntegrated M.Sc. Programmes 2026-27
(Deemed to be Universi1y under section 3 of UGC Ace, 1956) 
14. IMPORTANT DATES
Issue of Online Application 
Last date to apply 
28th January, 2026 (Wednesday) 
27th May, 2026 (Wednesday)
Result & Programme Choice preference...
```

# 7 Load LLM

In [111]:
# Load the tokenizer and model for FLAN-T5 Small
# 'google/flan-t5-small' is a compact yet capable language model for text generation.
print("Loading tokenizer for google/flan-t5-small...")
tokenizer = AutoTokenizer.from_pretrained("google/flan-t5-small")
print("Tokenizer loaded.")

print("Loading model for google/flan-t5-small... This might take a moment.")
model = AutoModelForSeq2SeqLM.from_pretrained("google/flan-t5-small")
print("Model loaded.")

# The Hugging Face pipeline for 'text2text-generation' is causing an 'Unknown task' error.
# We will directly use model.generate() in the 'ask' function instead.
print("Model and tokenizer (FLAN-T5 Small) loaded successfully for direct generation.")


Loading tokenizer for google/flan-t5-small...
Tokenizer loaded.
Loading model for google/flan-t5-small... This might take a moment.


Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


Model loaded.
Model and tokenizer (FLAN-T5 Small) loaded successfully for direct generation.


# 8 Retrieval Function

In [107]:
# Continue from previous cell, re-writing the retrieve_context function for clarity and correctness.

def retrieve_context(question: str, top_k: int = 4, score_threshold: float = 0.7) -> str:
    """
    Retrieves relevant context chunks from the FAISS vector database based on a question.
    Removes duplicate chunks and ignores chunks with poor similarity based on a score threshold.

    Args:
        question (str): The user's question.
        top_k (int): The number of most relevant chunks to retrieve initially.
        score_threshold (float): Similarity score threshold. Chunks with scores above this (less similar) are ignored.
                                 Lower score typically means higher similarity for FAISS with L2 distance.

    Returns:
        str: A combined string of unique and relevant context chunks.
    """
    # Perform similarity search to get documents and their scores.
    # FAISS 'similarity_search_with_score' typically returns (Document, score) tuples.
    # For 'all-MiniLM-L6-v2' and FAISS default, scores are usually L2 distance, where lower is better.
    results_with_scores = db.similarity_search_with_score(question, k=top_k)

    relevant_chunks_content = []
    seen_contents = set()  # To track unique chunk contents

    print(f"\nRetrieving context for question: '{question}'")
    print(f"Initial {len(results_with_scores)} chunks retrieved before filtering/deduplication:")

    # Diagnostic: Print all scores before filtering
    print("Diagnostic: Retrieved document scores (L2 distance, lower is better):")
    for i, (doc, score) in enumerate(results_with_scores):
        print(f"  [{i+1}] Score: {score:.4f}, Page: {doc.metadata.get('page')}, Content (first 50 chars): {doc.page_content[:50]}...")

    # Temporarily bypass the score_threshold for debugging to see all potential chunks
    # We will adjust score_threshold later based on these observed scores.
    for doc, score in results_with_scores:
        if doc.page_content not in seen_contents:
            relevant_chunks_content.append(doc.page_content)
            seen_contents.add(doc.page_content)
            print(f"  - ADDING CHUNK (Score: {score:.4f}, Page: {doc.metadata.get('page')}): {doc.page_content[:50]}...")
        else:
            print(f"  - Skipped duplicate chunk (Score: {score:.4f}, Page: {doc.metadata.get('page')}): {doc.page_content[:50]}...")

    if not relevant_chunks_content:
        print("No unique chunks found after initial retrieval (before score filtering). This is unexpected if results_with_scores was not empty.")
        return ""

    combined_context = "\n\n".join(relevant_chunks_content)
    print(f"\nFound {len(relevant_chunks_content)} unique chunks (score filtering temporarily bypassed).")

    return combined_context

# 9 RAG Answer Function

In [118]:
def ask(question: str) -> str:
    """
    Answers a question using Retrieval-Augmented Generation (RAG).
    Retrieves context from the FAISS database and uses FLAN-T5 Small to generate an answer.

    Args:
        question (str): The user's question.

    Returns:
        str: The generated answer or a predefined message if the answer is not found in context.
    """

    # 1. Retrieve context
    context = retrieve_context(question)

    if not context: # If no relevant context is found
        return "I couldn't find this information in the uploaded brochure."

    # 2. Construct a prompt for the LLM
    # The prompt guides the LLM on how to use the context and answer the question.
    prompt_template = f"""
    Based on the following context, answer the question concisely.
    If the answer is not in the context, state that you couldn't find the information.

    Context:
    {context}

    Question: {question}

    Answer:
    """

    # 3. Generate a concise answer using FLAN-T5 directly (bypassing pipeline due to task error)
    try:
        # Encode the prompt
        input_ids = tokenizer.encode(prompt_template, return_tensors='pt', max_length=512, truncation=True)

        # Generate response. Use model.generate() directly as pipeline was problematic.
        # max_new_tokens controls the length of the generated answer.
        output = model.generate(input_ids, max_new_tokens=100, num_beams=5, early_stopping=True)

        # Decode the generated tokens back to text
        generated_text = tokenizer.decode(output[0], skip_special_tokens=True).strip()

        # Post-process to check if the LLM hallucinated or didn't use the context properly.
        # This is a heuristic and can be improved. FLAN-T5 is designed to be extractive/abstractive
        # but can still generate outside context if the prompt isn't strong enough or context is too weak.
        # A simple check: if the response is too generic or implies lack of info, use fallback.
        # This part is crucial to prevent hallucination as per the requirement.

        # Heuristic for checking if the answer is 'not found' or a hallucination.
        # FLAN-T5 might generate short, generic answers or rephrase the 'not found' part of the prompt.
        # We need to ensure it specifically states the exact phrase if not found.
        if "couldn't find" in generated_text.lower() or "not provided" in generated_text.lower() or "information is not available" in generated_text.lower():
             return "I couldn't find this information in the uploaded brochure."

        # If the context is very small or the answer is very short and generic, it might be a hallucination
        # This part is tricky as a short answer might be correct.
        # For now, rely on the LLM's adherence to the prompt and the `retrieve_context` filtering.
        return generated_text

    except Exception as e:
        print(f"Error during LLM generation: {e}")
        return "An error occurred while generating the answer. Please try again."


# 10 Interactive Chatbot

In [119]:
print("Chatbot initialized. Type 'exit' or 'quit' to end the conversation.")

while True:
    user_question = input("\nUser: ")
    if user_question.lower() in ['exit', 'quit']:
        print("Chatbot: Goodbye!")
        break

    # Get the answer from the RAG system
    bot_answer = ask(user_question)
    print(f"Chatbot: {bot_answer}")


Chatbot initialized. Type 'exit' or 'quit' to end the conversation.

User: What is the hostel policy?

Retrieving context for question: 'What is the hostel policy?'
Initial 2 chunks retrieved before filtering/deduplication:
Diagnostic: Retrieved document scores (L2 distance, lower is better):
  [1] Score: 1.6992, Page: 14, Content (first 50 chars): VIT
® 
Vellore Institute of Technology lTntegrated...
  [2] Score: 1.7564, Page: 14, Content (first 50 chars): Director, PG Admissions 
Ve I lore Institute of Te...
  - ADDING CHUNK (Score: 1.6992, Page: 14): VIT
® 
Vellore Institute of Technology lTntegrated...
  - ADDING CHUNK (Score: 1.7564, Page: 14): Director, PG Admissions 
Ve I lore Institute of Te...

Found 2 unique chunks (score filtering temporarily bypassed).
Chatbot: The Registrar shall be the legal person in whose name the Institute may sue or be sued

User: How is seat allotment done?

Retrieving context for question: 'How is seat allotment done?'
Initial 2 chunks retrieved bef


KeyboardInterrupt



# 11 Sample Questions

In [120]:
sample_questions = [
    "What programmes are offered?",
    "What is the annual tuition fee?",
    "Who is eligible for Integrated M.Sc Physics?",
    "What documents are required during admission?",
    "How is seat allotment done?",
    "What are the important admission dates?",
    "What is the hostel policy?",
    "What are the programme highlights?",
    "Tell me about the campus life.",
    "What are the career opportunities after graduation?",
    "Is there any information about scholarships?",
    "How can I contact the admissions office?",
    "What is the last date to apply?"
]

print("Here are some sample questions you can try:\n")
for i, q in enumerate(sample_questions):
    print(f"{i+1}. {q}")

print("\nNow, you can use these questions in the interactive chatbot above or try your own!")


Here are some sample questions you can try:

1. What programmes are offered?
2. What is the annual tuition fee?
3. Who is eligible for Integrated M.Sc Physics?
4. What documents are required during admission?
5. How is seat allotment done?
6. What are the important admission dates?
7. What is the hostel policy?
8. What are the programme highlights?
9. Tell me about the campus life.
10. What are the career opportunities after graduation?
11. Is there any information about scholarships?
12. How can I contact the admissions office?
13. What is the last date to apply?

Now, you can use these questions in the interactive chatbot above or try your own!
